## Tutorial 3: Image segmentation using U-Net model

This notebook is a walkthrough of segmentation using a U-Net model in PyTorch.

This tutorial assumes no prior AI/ML background. We explain each idea before coding it.

Note - This is a version that should work well on your local computer/Google Colab using CPU. There are other variations that work well with GPUs.

Running this code on your local computer will take some time (about half an hour). 

## What you will learn

1. What a U-Net model is
2. What U-Net is made up of
3. How to prepare XCT images and masks for training, validation, and testing
4. How to train a U-Net segmentation model using PyTorch
5. How to evaluate model performance
<img src="./notebook_img/u-net_model_arch.png" width="600" alt="U-Net Architecture">

## Credits

These code chunks were initially adapted from https://github.com/usuyama/pytorch-unet and refined by GitHub Co-pilot.

## Points of contact

- Sameera Nalin Venkat (Post Doctorate RA C/William R. Wiley Postdoctoral Fellow): sameera.nalinvenkat@pnnl.gov
- Layton Washburn (Software Engineer): layton.washburn@pnnl.gov

Before we get started, let us review the core ideas of a neural network and what a U-Net model is!


### What is a neural network?

- Neural network (NN) can learn patterns without pre-defined rules unlike ML algorithms
- Consist of interconnected processing units called neurons, which are organized into layers
- Each layer takes info, transforms it, and passes it to the next layer

Early layers learn simple patterns such as edges, corners, and textures, whereas deeper layers learn more concepts such as shapes and object parts.

During training, NN compares its predictions to the correct answers (ground truth) and adjusts itself to make better predictions eventually.

In short, for image segmentation:

- Input is the raw image
- Processing involves many layers that learn useful patterns without pre-defined rules
- Output is a prediction of labels for each and every pixel

Here is the process of how a neural network operates

<div style="text-align: center; margin-top: 20px"><img src="notebook_img/nn_schematic.png" width="600" alt="Neural network schematic"></div>


### What is an epoch?

An **epoch** means the model has seen the **entire training dataset once**.

If you have 100 training images and use batch size 10:

- One batch = 10 images
- One epoch = 10 batches (until all 100 images are used once)

Models usually need many epochs, because they improve a little each time they go through the data.

### How the entire process happens in simple steps

1. Take one batch of images and masks
2. Run the model to make predictions
3. Compare predictions with ground truth using a loss function
4. Compute gradients (how each parameter should change)
5. Update model parameters using the optimizer
6. Repeat for all batches to finish one epoch
7. Check validation set performance, then start the next epoch

### Two key building blocks you will see in the model code

1. **Convolution (`Conv2d`)** - Think of this as a small moving window that scans the image
   - Looks for useful patterns such as edges, blobs, and textures

2. **ReLU (`ReLU`)** - After convolution, ReLU keeps positive values and turns negative values into 0
   - This helps the network focus on useful signals and learn complex patterns

Very simple analogy:

- Convolution = a feature detector
- ReLU = a filter that keeps strong helpful signals

Feel free to explore the Resources section of this notebook for more details.

### What is a U-Net model?

U-Net is a common architecture for segmentation tasks and it combines:

Here is how a structure of a U-Net model looks like:

<div style="text-align: center; margin-top: 20px"><img src="notebook_img/u-net_model_arch.png" width="600" alt="U-Net model schematic"></div>

There are some components that make up a U-Net model:

- An encoder that captures overall context and high-level features and shrinks the image while learning
- Bottleneck captures the compressed essence of the image
- A decoder that recovers finer spatial details by expanding image to full size, converting abstract features into pixel-level predictions
- Skip connections that pass finer features from encoder to decoder (think of them as bridges connecting encoder to decoder)

The main workflow of a U-Net model: Compress -> Extract Essence -> Expand + Restore Fine Details -> Accurate Predictions

All of this is accomplished using building blocks (or layers) of a neural network
- Feel free to refer to the Resources section if you are interested in the inner workings of neural networks and U-Net model

Similar to the random forest notebook, we will use intersection over union (IoU) for evaluating the model performance.

# 0) Clone the repository

In [ ]:
!git clone https://github.com/EMSL-MONet/Connect_and_Learn_2026 # Clone the repository to your colab runtiem environemnt

## 1) Data Overview and Preprocessing

In this section we will:

- Inspect a few raw image/mask pairs
- Convert annotation masks into binary labels
- Create train/validation/test splits 
- Build PyTorch `DataLoader`s (PyTorch is a Python library for building and training neural networks)

### What is a DataLoader?

A **DataLoader** is a PyTorch utility that feeds data to the model in mini-batches during training and evaluation.

Think of it this way:

- Dataset is what each item looks like (could be image or mask)
- DataLoader is how items are delivered to the model using batching, shuffling, and iteration



In [ ]:
# ----------------------------
# Notebook And Device Setup
# ----------------------------
# Make sure the reference U-Net repository is available locally
# This repo includes helper files such as `loss.py` that are used later
import os
import torch

if not os.path.exists("pytorch_unet.py"):
    # Clone once if the folder is not present yet
    if not os.path.exists("pytorch_unet"):
        !git clone https://github.com/usuyama/pytorch-unet.git

    # Move into the cloned project directory so local imports work
    %cd pytorch-unet

# Choose GPU if available or fall back to CPU
# Keeping CPU support helps in workshop settings and on local laptops
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

# Reminder for later cells
# Any model and tensors used for training or inference must be sent to `device`
# model = model.to(device)
# tensor = tensor.to(device)

In [ ]:
# ----------------------------
# Basic Imports
# ----------------------------
import os
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

# ----------------------------
# Dataset Preview Settings
# ----------------------------
# Paths to input images and binary masks
IMAGE_DIR = "../Connect_and_Learn_2026/XCT/data/raw_img"
MASK_DIR  = "../Connect_and_Learn_2026/XCT/data/otsu_masks"

# All images and masks are resized to a common size for batching
# PIL expects size as (width, height)
IMG_SIZE  = (256, 256)
NUM_PREVIEW = 3
NUM_CLASSES = 2  # Because we are looking at identifying pores from XCT images

# Read and sort file names so image and mask pairs align by index
image_files = sorted([f for f in os.listdir(IMAGE_DIR) if f.lower().endswith((".tif", ".tiff"))])
mask_files  = sorted([f for f in os.listdir(MASK_DIR)  if f.lower().endswith((".tif", ".tiff"))])
assert len(image_files) == len(mask_files) and len(image_files) > 0

# Load one RGB image from disk and resize it to IMG_SIZE
def load_img(path):
    return np.array(Image.open(path).convert("RGB").resize(IMG_SIZE))

# Load one mask as a single grayscale channel
# Nearest-neighbor resizing keeps class labels from being blended
def load_mask(path):
    return np.array(Image.open(path).convert("L").resize(IMG_SIZE, Image.NEAREST))

# Preview a few image and mask pairs to sanity-check the dataset
imgs  = [load_img(os.path.join(IMAGE_DIR, f)) for f in image_files[:NUM_PREVIEW]]
masks = [load_mask(os.path.join(MASK_DIR,  f)) for f in mask_files[:NUM_PREVIEW]]

fig, axes = plt.subplots(NUM_PREVIEW, 2, figsize=(7, 3 * NUM_PREVIEW))
for i in range(NUM_PREVIEW):
    axes[i, 0].imshow(imgs[i])
    axes[i, 0].set_title(image_files[i])
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks[i], cmap="gray")
    axes[i, 1].set_title(mask_files[i])
    axes[i, 1].axis("off")

plt.tight_layout()
plt.show()

# Helpful check for binary masks
# Values are often {0, 255} or {0, 1} before binarization
print("Mask unique values (sample):", np.unique(masks[0]))

In [ ]:
# ----------------------------
# Dataset And Dataloader Setup
# ----------------------------
import random
import os
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Fraction of data used for validation and test splits
# Generally 20%-30% of the image + mask pairs are reserved for validation and testing
VAL_SPLIT  = 0.15
TEST_SPLIT = 0.15

# Number of image and mask pairs sampled from the full dataset
# Increase this later if you want a larger training run
SAMPLE_N   = 100

class ImageMaskDataset(Dataset):
    """PyTorch Dataset returning (image_tensor, one_hot_mask)"""
    def __init__(self, image_paths, mask_paths, transform=None):
        self.image_paths = image_paths
        self.mask_paths  = mask_paths
        self.transform   = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        # Load the image and convert it to an RGB array with shape (H, W, 3)
        img = Image.open(self.image_paths[idx]).convert("RGB").resize(IMG_SIZE)
        img = np.array(img)

        # Load the mask as grayscale and binarize it to class values {0, 1}
        # Threshold 128 assumes bright pixels are foreground
        mask = Image.open(self.mask_paths[idx]).convert("L").resize(IMG_SIZE, Image.NEAREST)
        mask = (np.array(mask) > 128).astype(np.uint8)

        # Convert the mask to one-hot format expected by the model and loss
        # Shape becomes (NUM_CLASSES, H, W)
        one_hot = np.zeros((NUM_CLASSES, IMG_SIZE[0], IMG_SIZE[1]), dtype=np.float32)
        for c in range(NUM_CLASSES):
            one_hot[c] = (mask == c).astype(np.float32)

        if self.transform:
            img = self.transform(img)

        return img, one_hot

# Discover all files and keep sorted order for deterministic pairing
all_images = sorted([os.path.join(IMAGE_DIR, f) for f in os.listdir(IMAGE_DIR)])
all_masks  = sorted([os.path.join(MASK_DIR,  f) for f in os.listdir(MASK_DIR)])

assert len(all_images) == len(all_masks), \
    f"Mismatch: {len(all_images)} images vs {len(all_masks)} masks"

# Randomly sample a subset so this notebook runs faster 
random.seed(42)
indices    = random.sample(range(len(all_images)), SAMPLE_N)
all_images = [all_images[i] for i in indices]
all_masks  = [all_masks[i]  for i in indices]

print(f"Randomly selected {SAMPLE_N} pairs")

# Split the sampled data into train, validation, and test sets
n       = len(all_images)
n_test  = int(n * TEST_SPLIT)
n_val   = int(n * VAL_SPLIT)
n_train = n - n_val - n_test

train_imgs = all_images[:n_train]
val_imgs   = all_images[n_train : n_train + n_val]
test_imgs  = all_images[n_train + n_val:]

train_msks = all_masks[:n_train]
val_msks   = all_masks[n_train : n_train + n_val]
test_msks  = all_masks[n_train + n_val:]

print(f"Total: {n} | Train: {n_train} | Val: {n_val} | Test: {n_test}")

# Standard image normalization used by many CNN backbones
trans = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Build datasets and dataloaders for each split
train_set = ImageMaskDataset(train_imgs, train_msks, transform=trans)
val_set   = ImageMaskDataset(val_imgs,   val_msks,   transform=trans)
test_set  = ImageMaskDataset(test_imgs,  test_msks,  transform=trans)

image_datasets = {'train': train_set, 'val': val_set, 'test': test_set}

batch_size = 8

dataloaders = {
    'train': DataLoader(train_set, batch_size=batch_size, shuffle=True,  num_workers=0),
    'val':   DataLoader(val_set,   batch_size=batch_size, shuffle=False, num_workers=0),
    'test':  DataLoader(test_set,  batch_size=batch_size, shuffle=False, num_workers=0),
}

dataset_sizes = {x: len(image_datasets[x]) for x in image_datasets}
print(f"Dataset sizes: {dataset_sizes}")

In [ ]:
# ----------------------------
# Batch Preview Helpers
# ----------------------------
import torchvision.utils

# Undo normalization so a tensor can be shown as a regular image
def reverse_transform(inp):
    """Undo normalization for visualization"""
    inp = inp.numpy().transpose((1, 2, 0))
    mean = np.array([0.485, 0.456, 0.406])
    std = np.array([0.229, 0.224, 0.225])
    inp = std * inp + mean
    inp = np.clip(inp, 0, 1)
    inp = (inp * 255).astype(np.uint8)
    return inp

# Inspect one batch to verify tensor shapes
# inputs  -> (B, 3, H, W)
# masks   -> (B, NUM_CLASSES, H, W)
inputs, masks = next(iter(dataloaders['train']))
print("Input batch shape:", inputs.shape)
print("Mask batch shape:", masks.shape)

# Show one sample image from the batch after reversing normalization
plt.imshow(reverse_transform(inputs[3]))
plt.title("Example training image (after reverse normalization)")
plt.axis("off")

## 2) Define the U-Net Model

This implementation uses a **ResNet-18 encoder + U-Net style decoder**.

### Why this architecture?

- **ResNet-18 encoder** extracts meaningful features while reducing image size
- **U-Net decoder** upsamples those features back to image resolution
- **Skip connections** pass fine-grained details from encoder to decoder
- This combination helps predict **accurate pixel-level masks**

### Encoder-decoder intuition

1. Input image enters the encoder
2. Encoder compresses spatial size and learns high-level context
3. Decoder expands features back to the original resolution
4. Skip connections restore boundary and texture detail
5. Final layer predicts class scores for each pixel

### What the model outputs

Output shape is `(batch, NUM_CLASSES, H, W)`, where:

- `batch` is the number of images in one mini-batch
- `NUM_CLASSES` is the number of classes (here background vs pore)
- `H` and `W` are image height and width

The highest score at each pixel is taken as the predicted class label.

In [ ]:
# ----------------------------
# Model Definition
# ----------------------------
import torch
import torch.nn as nn
from torchvision import models

# Small helper block used throughout the decoder
def convrelu(in_channels, out_channels, kernel, padding):
    """Convenience block combining convolution and ReLU activation"""
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel, padding=padding),
        nn.ReLU(inplace=True),
    )

class ResNetUNet(nn.Module):
    """ResNet18 encoder and U-Net decoder for segmentation"""
    def __init__(self, n_class):
        super().__init__()

        # Encoder path based on ResNet18
        # weights=None keeps the notebook offline-friendly 
        self.base_model = models.resnet18(weights=None)
        self.base_layers = list(self.base_model.children())

        # Feature maps from multiple resolutions for skip connections
        self.layer0 = nn.Sequential(*self.base_layers[:3])      # (N, 64, H/2,  W/2)
        self.layer0_1x1 = convrelu(64, 64, 1, 0)
        self.layer1 = nn.Sequential(*self.base_layers[3:5])     # (N, 64, H/4,  W/4)
        self.layer1_1x1 = convrelu(64, 64, 1, 0)
        self.layer2 = self.base_layers[5]                       # (N, 128, H/8,  W/8)
        self.layer2_1x1 = convrelu(128, 128, 1, 0)
        self.layer3 = self.base_layers[6]                       # (N, 256, H/16, W/16)
        self.layer3_1x1 = convrelu(256, 256, 1, 0)
        self.layer4 = self.base_layers[7]                       # (N, 512, H/32, W/32)
        self.layer4_1x1 = convrelu(512, 512, 1, 0)

        # Decoder path upsamples features back to image resolution
        self.upsample = nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True)

        # Each decoder block merges an upsampled feature map with a skip connection
        self.conv_up3 = convrelu(256 + 512, 512, 3, 1)
        self.conv_up2 = convrelu(128 + 512, 256, 3, 1)
        self.conv_up1 = convrelu(64 + 256, 256, 3, 1)
        self.conv_up0 = convrelu(64 + 256, 128, 3, 1)

        # Full-resolution branch helps recover fine boundary details
        self.conv_original_size0 = convrelu(3, 64, 3, 1)
        self.conv_original_size1 = convrelu(64, 64, 3, 1)
        self.conv_original_size2 = convrelu(64 + 128, 64, 3, 1)

        # Final 1x1 convolution converts features into per-class logits
        self.conv_last = nn.Conv2d(64, n_class, 1)

    def forward(self, input):
        # Keep early high-resolution features from the raw input
        x_original = self.conv_original_size0(input)
        x_original = self.conv_original_size1(x_original)

        # Encoder forward pass
        layer0 = self.layer0(input)
        layer1 = self.layer1(layer0)
        layer2 = self.layer2(layer1)
        layer3 = self.layer3(layer2)
        layer4 = self.layer4(layer3)

        # Decoder step 1
        # Upsample the deepest features and merge with the layer3 skip connection
        layer4 = self.layer4_1x1(layer4)
        x = self.upsample(layer4)
        layer3 = self.layer3_1x1(layer3)
        x = torch.cat([x, layer3], dim=1)
        x = self.conv_up3(x)

        # Decoder step 2
        # Merge with the layer2 skip connection
        x = self.upsample(x)
        layer2 = self.layer2_1x1(layer2)
        x = torch.cat([x, layer2], dim=1)
        x = self.conv_up2(x)

        # Decoder step 3
        # Merge with the layer1 skip connection
        x = self.upsample(x)
        layer1 = self.layer1_1x1(layer1)
        x = torch.cat([x, layer1], dim=1)
        x = self.conv_up1(x)

        # Decoder step 4
        # Merge with the layer0 skip connection
        x = self.upsample(x)
        layer0 = self.layer0_1x1(layer0)
        x = torch.cat([x, layer0], dim=1)
        x = self.conv_up0(x)

        # Final fusion with the earliest full-resolution feature branch
        x = self.upsample(x)
        x = torch.cat([x, x_original], dim=1)
        x = self.conv_original_size2(x)

        # Return raw logits for each class at each pixel
        out = self.conv_last(x)
        return out

In [ ]:
# ----------------------------
# Model Initialization
# ----------------------------
import torch
import torch.nn as nn
from torchvision import models
import pytorch_unet

# Re-check device so this cell still works if run on its own
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device', device)

# Create a 2-class segmentation model and move it to the selected device
model = ResNetUNet(NUM_CLASSES)
model = model.to(device)

In [ ]:
# ----------------------------
# Display Model
# ----------------------------
# Show the full model structure in the notebook output
model

In [ ]:
# ----------------------------
# Model Summary
# ----------------------------
from torchsummary import summary

# Print a layer-by-layer summary for a sample input size
summary(model, input_size=(3, 224, 224))

In [ ]:
# ----------------------------
# Training And Metric Utilities
# ----------------------------
from collections import defaultdict
import torch
import torch.nn.functional as F
from loss import dice_loss

checkpoint_path = "checkpoint.pth"

# Compute foreground IoU from logits and one-hot targets
def calc_iou(pred, target, eps=1e-7):
    """Compute foreground IoU from logits and one-hot targets"""
    pred_labels = torch.argmax(pred, dim=1)
    target_labels = torch.argmax(target, dim=1)

    pred_fg = pred_labels == 1
    target_fg = target_labels == 1

    intersection = (pred_fg & target_fg).float().sum(dim=(1, 2))
    union = (pred_fg | target_fg).float().sum(dim=(1, 2))
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

# Compute the training loss and update running metrics
def calc_loss(pred, target, metrics, bce_weight=0.5):
    """
    Combined segmentation loss
    - BCE with logits gives pixel-wise supervision
    - Dice loss helps with overlap quality and class imbalance

    IoU is tracked separately as the main evaluation metric
    """
    bce = F.binary_cross_entropy_with_logits(pred, target)

    # Convert logits to probabilities before Dice computation
    pred_probs = torch.sigmoid(pred)
    dice = dice_loss(pred_probs, target)
    iou = calc_iou(pred, target)

    # Blend BCE and Dice into one scalar loss used for optimization
    loss = bce * bce_weight + dice * (1 - bce_weight)

    # Store totals multiplied by batch size so epoch averages are correct
    metrics['bce'] += bce.item() * target.size(0)
    metrics['iou'] += iou.item() * target.size(0)
    metrics['loss'] += loss.item() * target.size(0)

    return loss

# Print average metrics for one phase of an epoch
def print_metrics(metrics, epoch_samples, phase):
    """Print average metrics for one epoch phase"""
    outputs = []
    for k in metrics.keys():
        outputs.append("{}: {:4f}".format(k, metrics[k] / epoch_samples))
    print("{}: {}".format(phase, ", ".join(outputs)))

# Main training loop with validation and checkpoint saving
def train_model(model, optimizer, scheduler, num_epochs=25):
    """
    Standard training loop with validation and best-checkpoint saving
    Returns the best model by validation loss and a history dictionary for plots
    """
    best_loss = 1e10

    history = {
        'train_loss': [], 'val_loss': [],
        'train_bce':  [], 'val_bce':  [],
        'train_iou':  [], 'val_iou':  [],
    }

    for epoch in range(num_epochs):
        print('Epoch {}/{}'.format(epoch, num_epochs - 1))
        print('-' * 10)

        since = time.time()

        # Each epoch includes a training phase and a validation phase
        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()   # enable gradient updates
            else:
                model.eval()    # freeze dropout and batch norm behavior

            metrics = defaultdict(float)
            epoch_samples = 0

            for inputs, labels in dataloaders[phase]:
                inputs = inputs.to(device)
                labels = labels.to(device)

                optimizer.zero_grad()

                # Only compute gradients during the training phase
                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    loss = calc_loss(outputs, labels, metrics)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                epoch_samples += inputs.size(0)

            print_metrics(metrics, epoch_samples, phase)
            epoch_loss = metrics['loss'] / epoch_samples
            epoch_bce = metrics['bce'] / epoch_samples
            epoch_iou = metrics['iou'] / epoch_samples

            # Save epoch metrics so they can be plotted later
            history[f'{phase}_loss'].append(epoch_loss)
            history[f'{phase}_bce'].append(epoch_bce)
            history[f'{phase}_iou'].append(epoch_iou)

            if phase == 'train':
                scheduler.step()
                for param_group in optimizer.param_groups:
                    print("LR", param_group['lr'])

            # Keep the checkpoint with the best validation loss
            if phase == 'val' and epoch_loss < best_loss:
                print(f"saving best model to {checkpoint_path}")
                best_loss = epoch_loss
                torch.save(model.state_dict(), checkpoint_path)

        time_elapsed = time.time() - since
        print('{:.0f}m {:.0f}s'.format(time_elapsed // 60, time_elapsed % 60))

    print('Best val loss: {:4f}'.format(best_loss))

    model.load_state_dict(torch.load(checkpoint_path))
    return model, history

In [ ]:
# ----------------------------
# Training Configuration
# ----------------------------
import torch
import torch.optim as optim
from torch.optim import lr_scheduler
import time

# Create a fresh model instance for training
model = ResNetUNet(NUM_CLASSES).to(device)

# Optional transfer-learning setup
# Freeze encoder layers first to stabilize early training on smaller datasets
for l in model.base_layers:
    for param in l.parameters():
        param.requires_grad = False

# Optimize only trainable parameters such as the decoder and any unfrozen layers
optimizer_ft = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)

# Reduce the learning rate every `step_size` epochs
exp_lr_scheduler = lr_scheduler.StepLR(optimizer_ft, step_size=8, gamma=0.1)

# Train the model and collect metric history for plotting
model, history = train_model(model, optimizer_ft, exp_lr_scheduler, num_epochs=20)

In [ ]:
# ----------------------------
# Plot Training Curves
# ----------------------------
import matplotlib.pyplot as plt

# One x-axis value per completed epoch
epochs = range(1, len(history['train_loss']) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle('Training & Validation Metrics', fontsize=14)

# Each tuple controls one panel in the figure
# Format is (history_key_suffix, chart_title, train_color, val_color)
metric_cfg = [
    ('loss', 'Combined Loss (BCE + Dice)', 'tab:blue',   'tab:orange'),
    ('bce',  'Binary Cross-Entropy Loss',  'tab:green',  'tab:red'),
    ('iou',  'Intersection over Union',    'tab:purple', 'tab:brown'),
]

for ax, (key, title, tc, vc) in zip(axes, metric_cfg):
    ax.plot(epochs, history[f'train_{key}'], color=tc, marker='o', label='Train')
    ax.plot(epochs, history[f'val_{key}'],   color=vc, marker='s', linestyle='--', label='Val')
    ax.set_title(title)
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Value')
    ax.legend()
    ax.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# ----------------------------
# Test Set Evaluation
# ----------------------------
# Evaluate the trained model on the held-out test split
# Test data is not used for model selection during training
import torch
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

model.eval()

test_metrics = defaultdict(float)
test_samples = 0

with torch.no_grad():
    for inputs, labels in dataloaders['test']:
        inputs = inputs.to(device)
        labels = labels.to(device)

        outputs = model(inputs)
        calc_loss(outputs, labels, test_metrics)
        test_samples += inputs.size(0)

print("=== Test Set Results ===")
print_metrics(test_metrics, test_samples, "test")

In [ ]:
# ----------------------------
# Visualize Test Predictions
# ----------------------------
# Show images, ground-truth masks, and model predictions side by side
NUM_VISUAL = 4  # number of rows to display

# Undo normalization so tensors can be displayed as normal images
def reverse_transform(inp):
    """Undo normalization so tensors are displayable as images"""
    inp   = inp.numpy().transpose((1, 2, 0))
    mean  = np.array([0.485, 0.456, 0.406])
    std   = np.array([0.229, 0.224, 0.225])
    inp   = std * inp + mean
    inp   = np.clip(inp, 0, 1)
    return (inp * 255).astype(np.uint8)

# Convert a one-hot or probability mask into a simple black-and-white RGB image
def masks_to_colorimg(mask):
    """Convert one-hot or probability mask to a black-and-white RGB visualization"""
    colors = np.array([
        [0,   0,   0],    # class 0 = background
        [255, 255, 255],  # class 1 = foreground
    ], dtype=np.uint8)
    return colors[np.argmax(mask, axis=0)]

model.eval()

# Grab one batch from the test loader for quick visualization
inputs, labels = next(iter(dataloaders['test']))
inputs = inputs.to(device)

with torch.no_grad():
    preds = model(inputs)
    preds = torch.sigmoid(preds).cpu().numpy()

inputs = inputs.cpu()
labels = labels.numpy()

n_show = min(NUM_VISUAL, inputs.size(0))
fig, axes = plt.subplots(n_show, 3, figsize=(10, n_show * 3))
fig.suptitle("Test Predictions\n(Left: Image | Middle: Ground Truth | Right: Prediction)", fontsize=12)

for i in range(n_show):
    axes[i, 0].imshow(reverse_transform(inputs[i]))
    axes[i, 0].set_title("Image")
    axes[i, 0].axis("off")

    axes[i, 1].imshow(masks_to_colorimg(labels[i]))
    axes[i, 1].set_title("Ground Truth")
    axes[i, 1].axis("off")

    axes[i, 2].imshow(masks_to_colorimg(preds[i]))
    axes[i, 2].set_title("Prediction")
    axes[i, 2].axis("off")

plt.tight_layout()
plt.show()

## 3) Suggested  next experiments

Great job completing this U-Net notebook! To improve results, try one change at a time and compare plots/visual outputs:

- **Learning rate**: test values like `1e-3`, `3e-4`, `1e-4`
- **Scheduler**: change `step_size` or try cosine scheduling
- **Loss balance**: adjust `bce_weight` in `calc_loss`
- **Backbone freezing**: unfreeze more encoder layers after a few epochs
- **Batch size / image size**: larger values may improve quality if memory allows

Suggestion - Keep a small experiment log (settings + final val/test metrics) so you can see what actually helped.

## Helpful resources

No worries if the details of U-Net model are not clear yet. We recommend that you check out the following resources to learn more:

- Introduction to image segmentation: https://www.youtube.com/watch?v=onWJQY5oFhs&list=PL2zRqk16wsdop2EatuowXBX5C-r2FdyNt
- CNN overview: https://www.youtube.com/watch?v=QzY57FaENXg 
- A few more math details behind CNN: https://www.youtube.com/watch?v=KuXjwB4LzSA

Happy learning!